In [3]:
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer                            # fill in missing values
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Making our initial data variables

In [7]:
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/ames'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
solns = pd.read_csv(f'{DATA_DIR}/solns.csv')      #SalePrice for every test row

# Features (X) and target (y)
X_train = train.drop(columns=['Id', 'SalePrice'])
y_train = train['SalePrice']
X_test  = test.drop(columns=['Id'])
y_test = solns.sort_values('Id')['SalePrice'].to_numpy()

print(f'train features: {X_train.shape}   test features: {X_test.shape}   test answers: {y_test.shape}')

train features: (1200, 79)   test features: (260, 79)   test answers: (260,)


# making functions to streamline our code *it got too large otherwise

In [10]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

  #rmse built into a function, honestly could've just imported root_mean_squared_error function

folds = KFold(n_splits=5, shuffle=True, random_state=0)   # folding data but standardized

results = []
predictions = {}

def evaluate(name, model, X_tr=X_train, X_te=X_test):
    """Fit `model`; report cross-validated RMSE on train and RMSE on test."""
    # 5 fold comes in here means that it splits train into fifths, taking 4/5 and testing it against the remaining 1/5, in this case repeated 5 times so it can be a little more accurate
    cv_rmse = -cross_val_score(model, X_tr, y_train, cv=folds,
                               scoring='neg_root_mean_squared_error').mean()

    model.fit(X_tr, y_train)
    preds = model.predict(X_te)
    test_rmse = rmse(y_test, preds)

    results.append({'model': name, 'CV RMSE (train)': round(cv_rmse), 'test RMSE': round(test_rmse)})
    predictions[name] = preds
    print(f'{name}\n   CV RMSE: ${cv_rmse:,.0f}    test RMSE: ${test_rmse:,.0f}')
    return model

MODEL ONE USING ONE FEATURE WHICH WE THOUGHT WAS GOOD

In [11]:
# One feature linear regressionevaluate('1. Linear: GrLivArea only', LinearRegression(),
         X_tr=X_train[['GrLivArea']], X_te=X_test[['GrLivArea']]);

1. Linear: GrLivArea only
   CV RMSE: $54,292    test RMSE: $63,777


MODEL TWO USING EVERY FEATURE TO CALCULATE RMSE

In [13]:
# Split the column names by type so we can turn categorical columns into something quantifiable
numeric_cols     = X_train.select_dtypes(include='number').columns.tolist()
categorical_cols = X_train.select_dtypes(exclude='number').columns.tolist()
print(f'{len(numeric_cols)} numeric columns, {len(categorical_cols)} categorical columns')

preprocess = ColumnTransformer([
    # fill blanks with the column median
    ('num', SimpleImputer(strategy='median'), numeric_cols),
    # categorical columns we want to fill blanks with 'None', then one-hot encode
    ('cat', make_pipeline(SimpleImputer(strategy='constant', fill_value='None'),
                          OneHotEncoder(handle_unknown='ignore')), categorical_cols),
])

evaluate('2. Linear: all features', make_pipeline(preprocess, LinearRegression()));

36 numeric columns, 43 categorical columns
2. Linear: all features
   CV RMSE: $30,095    test RMSE: $46,485


MODEL 5 USING TREES


In [ ]:
# Turning data values into data type category because the HistGradiantBoosting Regressor takes in category data type not floats, booleans, or numberical ones
X_train_cat = X_train.copy();  X_train_cat[categorical_cols] = X_train_cat[categorical_cols].astype('category')
X_test_cat  = X_test.copy();   X_test_cat[categorical_cols]  = X_test_cat[categorical_cols].astype('category')

def boosted_trees(max_features=1.0, min_samples_leaf=20):
    return TransformedTargetRegressor(
        regressor=HistGradientBoostingRegressor(
            categorical_features='from_dtype',   # use the category columns
            learning_rate=0.05,                  # how much each tree corrects the previous ones
            max_iter=600,                        # number of trees made
            max_leaf_nodes=15,                   # size of each tree, generalize better
            l2_regularization=1.0,                # accounting for outliers making the data weird
            max_features=max_features,           # fraction of columns each tree uses, so one equals 100% )
            min_samples_leaf=min_samples_leaf,   # every leaf must have at least this many houses
            random_state=0),
        func=np.log1p, inverse_func=np.expm1)    # log price so its percentage and also removes negatives

# trying six different versions of the HistGradiantRegressor, appending it to our sweep list with the RMSE and its details to pick the best with AKA index 0 because its sorted based on lowest RMSE
sweep = []
for max_features in [1.0, 0.5, 0.3]:
    for min_samples_leaf in [10, 20]:
        cv = -cross_val_score(boosted_trees(max_features, min_samples_leaf), X_train_cat, y_train,
                              cv=folds, scoring='neg_root_mean_squared_error').mean()
        sweep.append({'max_features': max_features, 'min_samples_leaf': min_samples_leaf, 'CV RMSE': round(cv)})
sweep = pd.DataFrame(sweep).sort_values('CV RMSE', ignore_index=True)
print(sweep.to_string(index=False))

best = sweep.iloc[0]
evaluate('5. Gradient boosting (settings by CV)',
         boosted_trees(best['max_features'], int(best['min_samples_leaf'])), X_tr=X_train_cat, X_te=X_test_cat);